In [0]:
# ============================================================
# EVALUATION NOTEBOOK — Setup + inspect the ground truth
# ============================================================
from pyspark.sql import functions as F

# Load the cleaned Dermstore data (has the real concern labels)
df_derm = spark.table("dermstore.dermstoreschema.silver_dermstore")

# The ground-truth field: skin_type_and_concerns (human-assigned, ~42% filled)
# Let's see what it actually looks like for products that HAVE it.
labeled = df_derm.filter(
    F.col("skin_type_and_concerns").isNotNull() &
    (F.trim(F.col("skin_type_and_concerns")) != "")
)

print("Products WITH ground-truth labels:", labeled.count())
print("Total products:", df_derm.count())
print("\nExamples of the raw ground-truth labels:")
labeled.select("product_id", "skin_type_and_concerns").show(10, truncate=100)

Products WITH ground-truth labels: 50
Total products: 120

Examples of the raw ground-truth labels:
+----------+----------------------------------------------------------------------------------------------------+
|product_id|                                                                              skin_type_and_concerns|
+----------+----------------------------------------------------------------------------------------------------+
|  12905594|Acne: BlackheadsAging Skin: Loss of FirmnessApplication Area: FaceIdeal for these Concerns: Acne,...|
|  11271963|Application Area: Hair, ScalpIdeal for these Concerns: Damaged Hair, Free Radical Damage, Frizzy ...|
|  12902323|Application Area: HairIdeal for these Concerns: Damaged Hair, Frizzy HairHair Care: ShampooHair T...|
|  12905883|Application Area: HairColor Family: RedIdeal for these Concerns: Damaged HairHair Care: Hair Colo...|
|  11287441|Aging Skin: Loss of FirmnessApplication Area: Body, Decollete, FaceBath and Body: Lotions 

In [0]:
# ============================================================
# EVALUATION — Step 1: Parse the ground-truth concerns
# ============================================================
# The labels contain "Ideal for these Concerns: X, Y, Z" — we extract
# that part. Then we map their wording to OUR concern vocabulary.

# 1. Extract the text after "Ideal for these Concerns:" up to the next
#    capitalized category header (e.g. "Application Area:", "Skin Care:").
#    regexp_extract pulls out the matched group.
labeled = labeled.withColumn(
    "concerns_raw",
    F.regexp_extract(
        F.col("skin_type_and_concerns"),
        r"Ideal for these Concerns:\s*([^A-Z]*(?:[A-Z][a-z]+(?:\s[A-Z][a-z]+)*,?\s*)*?)(?=[A-Z][a-z]+:|$)",
        1
    )
)

# See what we extracted (raw, before mapping)
labeled.select("product_id", "concerns_raw").show(15, truncate=80)

+----------+-----------------------------------------------------------+
|product_id|                                               concerns_raw|
+----------+-----------------------------------------------------------+
|  12905594|                                                           |
|  11271963|        Damaged Hair, Free Radical Damage, Frizzy HairHair |
|  12902323|                             Damaged Hair, Frizzy HairHair |
|  12905883|                                          Damaged HairHair |
|  11287441|                                                           |
|  11289620|                                                           |
|  13175647|                                                           |
|  11307158|                                                           |
|  12901687|                                        Free Radical Damage|
|  11486957|                                                Large Pores|
|  11146992|                                       

In [0]:
# ============================================================
# EVALUATION — Step 1 (simpler, more robust): keyword-match concerns
# ============================================================
# Instead of fragile parsing, directly check which known concerns appear
# in each ground-truth label. We map their wording to ours.

# Map: phrases that appear in the ground truth -> our concern vocabulary
GT_CONCERN_MAP = {
    "acne": "acne",
    "blackhead": "blackheads",
    "oily": "oily skin",
    "large pore": "large pores",
    "pore": "large pores",
    "dark spot": "hyperpigmentation",
    "hyperpigmentation": "hyperpigmentation",
    "uneven skin tone": "hyperpigmentation",
    "discoloration": "hyperpigmentation",
    "fine lines": "aging",
    "wrinkle": "aging",
    "loss of firmness": "firmness",
    "firmness": "firmness",
    "dry": "dryness",
    "dehydrat": "dehydration",
    "dull": "dullness",
    "redness": "redness",
    "dark circle": "dark circles",
    "texture": "texture",
}

# For each labeled product, find which of our concerns its label mentions
from pyspark.sql.types import ArrayType, StringType

def extract_gt_concerns(label):
    if not label:
        return []
    label_low = label.lower()
    found = []
    for phrase, concern in GT_CONCERN_MAP.items():
        if phrase in label_low and concern not in found:
            found.append(concern)
    return found

extract_udf = F.udf(extract_gt_concerns, ArrayType(StringType()))
labeled = labeled.withColumn("gt_concerns", extract_udf(F.col("skin_type_and_concerns")))

# See the ground-truth concerns we extracted
labeled.select("product_id", "gt_concerns").show(20, truncate=60)

+----------+------------------------------------------------------------+
|product_id|                                                 gt_concerns|
+----------+------------------------------------------------------------+
|  12905594|[acne, blackheads, oily skin, large pores, firmness, dryn...|
|  11271963|                                                   [dryness]|
|  12902323|                                        [oily skin, dryness]|
|  12905883|                                                          []|
|  11287441|           [acne, oily skin, firmness, dryness, dehydration]|
|  11289620|                                  [acne, oily skin, dryness]|
|  13175647|                 [acne, oily skin, aging, firmness, dryness]|
|  11307158|                                                          []|
|  12901687|                                                          []|
|  11486957|                                               [large pores]|
|  11146992|                          

In [0]:
# ============================================================
# EVALUATION — Inspect full labels to find the exact structure
# ============================================================
# Print a few complete labels so we can see exactly how
# "Ideal for these Concerns:" is bounded by the next category.
for row in labeled.select("product_id", "skin_type_and_concerns").take(5):
    print(f"--- Product {row['product_id']} ---")
    print(row['skin_type_and_concerns'])
    print()

--- Product 12905594 ---
Acne: BlackheadsAging Skin: Loss of FirmnessApplication Area: FaceIdeal for these Concerns: Acne, Large Pores, OilinessProfessional Tools and Devices: Cleansing DevicesSkin Care: Professional Tools and DevicesSkin Type: Acne-Prone Skin, Combination Skin, Dry Skin, Mature Skin, Normal Skin, Oily Skin, Sensitive Skin

--- Product 11271963 ---
Application Area: Hair, ScalpIdeal for these Concerns: Damaged Hair, Free Radical Damage, Frizzy HairHair Care: ShampooHair Type: Coarse Hair, Curly Hair, Dry or Damaged HairIngredient: Aloe Vera, Amino Acids, Antioxidants, Shea Butter, Vitamin C, Vitamin EPreferences: Certified Cruelty-Free, Gluten-Free, Natural, Paraben-Free, Sulfate-FreeShampoo: Smoothing Shampoo

--- Product 12902323 ---
Application Area: HairIdeal for these Concerns: Damaged Hair, Frizzy HairHair Care: ShampooHair Type: Coarse Hair, Color-Treated Hair, Curly Hair, Dry or Damaged Hair, Fine Hair, Normal Hair, Oily HairIngredient: Essential Oils, Herbal E

In [0]:
# ============================================================
# EVALUATION — Step 1 (Option A): Extract ONLY the curated concerns
# ============================================================
# Grab the text after "Ideal for these Concerns:" up to the next
# category header (pattern: "Word:" or "Words Words:").
# This is the CURATED concern list — trustworthy ground truth.

import re
from pyspark.sql.types import ArrayType, StringType

# Map the ground-truth wording -> our concern vocabulary
GT_CONCERN_MAP = {
    "acne": "acne",
    "blackheads": "blackheads",
    "oiliness": "oily skin",
    "oily skin": "oily skin",
    "large pores": "large pores",
    "dark spots": "hyperpigmentation",
    "hyperpigmentation": "hyperpigmentation",
    "uneven skin tone": "hyperpigmentation",
    "discoloration": "hyperpigmentation",
    "dullness": "dullness",
    "fine lines and wrinkles": "aging",
    "wrinkles": "aging",
    "aging skin": "aging",
    "loss of firmness": "firmness",
    "dryness and dehydration": "dryness",
    "dryness": "dryness",
    "redness": "redness",
    "dark circles": "dark circles",
    "uneven texture": "texture",
}

def extract_curated_concerns(label):
    if not label:
        return []
    # Find the "Ideal for these Concerns:" section, stopping at the next
    # "CapitalizedWord(s):" category header.
    m = re.search(r"Ideal for these Concerns:\s*(.*?)(?=[A-Z][a-zA-Z]*(?:\s[A-Z][a-zA-Z]*)*:|$)", label)
    if not m:
        return []
    raw_concerns = m.group(1)
    # Split the concern list on commas
    items = [c.strip().lower() for c in raw_concerns.split(",") if c.strip()]
    # Map each to our vocabulary (skip ones not in our skincare map, e.g. hair)
    mapped = []
    for item in items:
        if item in GT_CONCERN_MAP:
            concern = GT_CONCERN_MAP[item]
            if concern not in mapped:
                mapped.append(concern)
    return mapped

extract_udf = F.udf(extract_curated_concerns, ArrayType(StringType()))
labeled = labeled.withColumn("gt_concerns", extract_udf(F.col("skin_type_and_concerns")))

# Show the clean ground truth
labeled.select("product_id", "gt_concerns").show(25, truncate=70)

+----------+----------------------------+
|product_id|                 gt_concerns|
+----------+----------------------------+
|  12905594|         [acne, large pores]|
|  11271963|                          []|
|  12902323|                          []|
|  12905883|                          []|
|  11287441|            [aging, dryness]|
|  11289620|                          []|
|  13175647|                          []|
|  11307158|                          []|
|  12901687|                          []|
|  11486957|                          []|
|  11146992|                          []|
|  11370322|         [hyperpigmentation]|
|  11287898|                          []|
|  12902951|                          []|
|  11837067|[dryness, hyperpigmentation]|
|  12918861|                   [redness]|
|  12535727|                          []|
|  11582931|                          []|
|  11689517|                          []|
|  10302720|                          []|
|  12902295|                   [dr

In [0]:
# How many products have NON-EMPTY ground truth after clean parsing?
usable = labeled.filter(F.size(F.col("gt_concerns")) > 0)
print("Products with usable ground truth:", usable.count())
print("\nThe usable ground-truth set:")
usable.select("product_id", "gt_concerns").show(50, truncate=70)

Products with usable ground truth: 18

The usable ground-truth set:
+----------+----------------------------+
|product_id|                 gt_concerns|
+----------+----------------------------+
|  12905594|         [acne, large pores]|
|  11287441|            [aging, dryness]|
|  11370322|         [hyperpigmentation]|
|  11837067|[dryness, hyperpigmentation]|
|  12918861|                   [redness]|
|  12902295|                   [dryness]|
|  11434924|              [dark circles]|
|  12902815|                   [dryness]|
|  11370875|          [dryness, redness]|
|  11291410|                     [aging]|
|  11709479|                   [redness]|
|  11207519|                     [aging]|
|  11912265|        [acne, dark circles]|
|  11443354|                      [acne]|
|  11370452|                     [aging]|
|  11296433|                      [acne]|
|  11329109|                   [dryness]|
|  11287247|                     [aging]|
+----------+----------------------------+



In [0]:
# ============================================================
# EVALUATION — Step 2: Compare our tags vs ground truth
# ============================================================
# For the 18 products with clean ground truth, compare what OUR system
# tagged against the real labels. Compute precision & recall per product.

# Get our system's concern tags
our_tags = spark.table("dermstore.dermstoreschema.gold_product_concerns") \
    .groupBy("product_id").agg(F.collect_list("concern").alias("our_concerns"))

# Join ground truth with our tags (only the 18 usable products)
eval_df = usable.select("product_id", "gt_concerns").join(our_tags, "product_id", "left")

# Compute precision & recall for each product
import pandas as pd
eval_pdf = eval_df.toPandas()

rows = []
for _, r in eval_pdf.iterrows():
    gt = set(r["gt_concerns"]) if r["gt_concerns"] is not None else set()
    ours = set(r["our_concerns"]) if r["our_concerns"] is not None else set()
    if not gt:
        continue
    overlap = gt & ours          # concerns we got right
    # precision = of what WE tagged, how much was correct
    precision = len(overlap) / len(ours) if ours else 0.0
    # recall = of the TRUE concerns, how many we found
    recall = len(overlap) / len(gt) if gt else 0.0
    rows.append({
        "product_id": r["product_id"],
        "ground_truth": sorted(gt),
        "our_tags": sorted(ours),
        "matched": sorted(overlap),
        "precision": round(precision, 2),
        "recall": round(recall, 2),
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))

print(f"\n--- SPOT-CHECK SUMMARY ({len(results)} products) ---")
print(f"Average recall:    {results['recall'].mean():.2f}  (of true concerns, how many we found)")
print(f"Average precision: {results['precision'].mean():.2f}  (of our tags, how many were correct)")

product_id                 ground_truth                                                                                                        our_tags             matched  precision  recall
  12905594          [acne, large pores]                                                                                                              []                  []       0.00     0.0
  11287441             [aging, dryness]                                                                                          [dehydration, dryness]           [dryness]       0.50     0.5
  11370322          [hyperpigmentation]                                                                                                              []                  []       0.00     0.0
  11837067 [dryness, hyperpigmentation]                                                    [aging, dark circles, dullness, firmness, hyperpigmentation] [hyperpigmentation]       0.20     0.5
  12918861                    [redness]      

In [0]:
# ============================================================
# EVALUATION — Option A: Re-tune tagging (recover recall + protect precision)
# ============================================================
# Eval revealed: low recall (empty tags) AND low precision (over-tagging).
# Fix: relax prominence cutoff (15->25) to recover recall, AND require
# 2+ matching ingredients per concern to protect precision.

from pyspark.sql import functions as F

# Reload needed tables
df_derm = spark.table("dermstore.dermstoreschema.silver_dermstore")
kb = spark.table("dermstore.dermstoreschema.concern_ingredient_kb")
skincare_products = spark.table("dermstore.dermstoreschema.gold_products").select("product_id")

# 1. Explode ingredients with position
derm_pos = df_derm.select(
    "product_id",
    F.posexplode("ingredients_list").alias("position", "ingredient")
)

# 2. RELAXED prominence cutoff: first 25 (was 15) -> recovers recall
PROMINENCE_CUTOFF = 25
derm_prominent = derm_pos.filter(F.col("position") < PROMINENCE_CUTOFF)
derm_prominent = derm_prominent.withColumn(
    "ingredient_clean", F.upper(F.trim(F.col("ingredient")))
)

# 3. Keep skincare products, join to knowledge base
derm_prominent = derm_prominent.join(skincare_products, "product_id")
matches = derm_prominent.join(
    kb,
    derm_prominent.ingredient_clean == kb.ingredient,
    how="inner"
).select("product_id", "concern", "ingredient_clean").distinct()

# 4. NEW precision rule: require 2+ DISTINCT matching ingredients per concern
concern_evidence = matches.groupBy("product_id", "concern").agg(
    F.countDistinct("ingredient_clean").alias("num_matching_ingredients")
)
MIN_INGREDIENTS_PER_CONCERN = 2
product_concerns_v3 = concern_evidence.filter(
    F.col("num_matching_ingredients") >= MIN_INGREDIENTS_PER_CONCERN
).select("product_id", "concern")

# Compare tag counts
print("New total tags (v3):", product_concerns_v3.count())
print("\nConcerns-per-product distribution:")
product_concerns_v3.groupBy("product_id").count() \
    .groupBy("count").count().orderBy("count").show()

New total tags (v3): 68

Concerns-per-product distribution:
+-----+-----+
|count|count|
+-----+-----+
|    1|    3|
|    2|   24|
|    3|    3|
|    4|    2|
+-----+-----+



In [0]:
# Save v3 tags (overwrite the gold table)
product_concerns_v3.write.mode("overwrite") \
    .saveAsTable("dermstore.dermstoreschema.gold_product_concerns")
print("Saved v3 tags.")

Saved v3 tags.


In [0]:
# ============================================================
# EVALUATION — Re-measure with v3 tags (THE PAYOFF)
# ============================================================
import pandas as pd

# Our NEW tags
our_tags = spark.table("dermstore.dermstoreschema.gold_product_concerns") \
    .groupBy("product_id").agg(F.collect_list("concern").alias("our_concerns"))

# Join ground truth (the 18 usable products) with new tags
eval_df = usable.select("product_id", "gt_concerns").join(our_tags, "product_id", "left")
eval_pdf = eval_df.toPandas()

rows = []
for _, r in eval_pdf.iterrows():
    gt = set(r["gt_concerns"]) if r["gt_concerns"] is not None else set()
    ours = set(r["our_concerns"]) if r["our_concerns"] is not None else set()
    if not gt:
        continue
    overlap = gt & ours
    precision = len(overlap) / len(ours) if ours else 0.0
    recall = len(overlap) / len(gt) if gt else 0.0
    rows.append({"product_id": r["product_id"], "ground_truth": sorted(gt),
                 "our_tags": sorted(ours), "matched": sorted(overlap),
                 "precision": round(precision, 2), "recall": round(recall, 2)})

results_v3 = pd.DataFrame(rows)
print(results_v3.to_string(index=False))
print(f"\n--- v3 RESULTS ({len(results_v3)} products) ---")
print(f"Average recall:    {results_v3['recall'].mean():.2f}")
print(f"Average precision: {results_v3['precision'].mean():.2f}")
print(f"\n(Compare to v2: recall 0.28, precision 0.10)")

product_id                 ground_truth                      our_tags   matched  precision  recall
  12905594          [acne, large pores]                            []        []        0.0     0.0
  11287441             [aging, dryness]                            []        []        0.0     0.0
  11370322          [hyperpigmentation]                            []        []        0.0     0.0
  11837067 [dryness, hyperpigmentation]                            []        []        0.0     0.0
  12918861                    [redness]                            []        []        0.0     0.0
  12902295                    [dryness]                            []        []        0.0     0.0
  11434924               [dark circles] [aging, dehydration, dryness]        []        0.0     0.0
  12902815                    [dryness]        [dehydration, dryness] [dryness]        0.5     1.0
  11370875           [dryness, redness]                            []        []        0.0     0.0
  11291410

In [0]:
# ============================================================
# EVALUATION — Revert to v2 (eval confirmed it was best)
# ============================================================
# v3's strict rule made things worse. Restore v2: prominence cutoff 15,
# no minimum-ingredient rule. (This was the FIX C version.)
from pyspark.sql import functions as F

df_derm = spark.table("dermstore.dermstoreschema.silver_dermstore")
kb = spark.table("dermstore.dermstoreschema.concern_ingredient_kb")
skincare_products = spark.table("dermstore.dermstoreschema.gold_products").select("product_id")

derm_pos = df_derm.select(
    "product_id",
    F.posexplode("ingredients_list").alias("position", "ingredient")
)
# v2 settings: cutoff 15, simple match (no 2+ rule)
derm_prominent = derm_pos.filter(F.col("position") < 15) \
    .withColumn("ingredient_clean", F.upper(F.trim(F.col("ingredient")))) \
    .join(skincare_products, "product_id")

product_concerns_v2 = derm_prominent.join(
    kb, derm_prominent.ingredient_clean == kb.ingredient, how="inner"
).select("product_id", "concern").distinct()

# Save it back as the gold table
product_concerns_v2.write.mode("overwrite") \
    .saveAsTable("dermstore.dermstoreschema.gold_product_concerns")

print("Reverted to v2. Total tags:", product_concerns_v2.count())

Reverted to v2. Total tags: 267
